# MiniGPT from scratch — Colab runner (timeout-safe)
Runs everything end to end: tests → architecture evidence → overfit gate → training → samples → attention maps → ablations → API check.

**Timeout-safe design:** `outputs/` and `checkpoints/` live on Google Drive, and training/ablations **resume automatically** from the last saved step.
If Colab disconnects: reconnect, run cells 1–2 (setup + Drive) again, then re-run the cell that was running. It continues where it stopped.

**First:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
REPO_URL = "https://github.com/adeena43/Small-Language-Model.git"
QUICK = False            # True = short smoke run (a few minutes)
MAIN_STEPS = 5000        # steps for the main model
ABLATION_STEPS = 2000    # steps per ablation run
ABLATION_SMALL = True    # True = ablations use a smaller model (d_model 128) so they finish faster; document this in the report

In [ ]:
# ---- CELL 1: setup (re-run after every disconnect) ----
import os, torch
if not os.path.exists("Small-Language-Model"):
    !git clone $REPO_URL Small-Language-Model
%cd Small-Language-Model
!git pull -q
!pip install -q -r requirements-train.txt
from IPython.display import Image, display
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ---- CELL 2: Google Drive (re-run after every disconnect) ----
# outputs/ and checkpoints/ are stored on Drive, so nothing is lost when the runtime dies.
from google.colab import drive
drive.mount('/content/drive')
P = "/content/drive/MyDrive/minigpt_runs"
os.makedirs(P + "/outputs", exist_ok=True); os.makedirs(P + "/checkpoints", exist_ok=True)
!cp -rn outputs/. {P}/outputs/ 2>/dev/null
!cp -rn checkpoints/. {P}/checkpoints/ 2>/dev/null
!rm -rf outputs checkpoints
!ln -s {P}/outputs outputs
!ln -s {P}/checkpoints checkpoints
!ls -la | grep -E "outputs|checkpoints"
!ls checkpoints

## 1. Unit tests

In [ ]:
!python -m pytest -q

## 2. Architecture evidence (Figures 1–5)

In [ ]:
!python experiments/hand_calculation.py
!python experiments/shape_trace.py
!python experiments/positional_encoding_viz.py
!python experiments/architecture_diagram.py
for f in ["fig1_hand_attention", "fig2_causal_mask", "fig4_positional_encoding", "architecture_diagram"]:
    display(Image(f"outputs/figures/{f}.png"))

## 3. Why Transformers (sequential vs parallel) and MLM-vs-CLM comparison

In [ ]:
!python experiments/why_transformers.py
!python experiments/mask_comparison.py
display(Image("outputs/figures/why_transformers.png")); display(Image("outputs/figures/fig6c_mlm_vs_clm_training.png"))

## 4. Debugging gate: overfit a tiny batch (loss must collapse)

In [ ]:
!python -m src.train --overfit_tiny

## 5. Train the main MiniGPT (resumable: if it disconnects, just re-run this cell)

In [ ]:
steps = 300 if QUICK else MAIN_STEPS
eval_i = 250 if steps >= 1000 else 50
samp_i = 1000 if steps >= 1000 else 100
!python -m src.train --resume --set train.max_steps={steps} train.eval_interval={eval_i} train.sample_interval={samp_i}

In [ ]:
import json
display(Image("outputs/figures/main_loss_curve.png"))
print(json.dumps(json.load(open("outputs/logs/main_summary.json")), indent=2)[:1800])

## 6. Generation (fixed prompts), evaluation, attention maps

In [ ]:
!python -m src.generate --fixed-prompts --max-new-tokens 200
!python experiments/evaluate.py --checkpoint checkpoints/best.pt --device cuda
!python experiments/attention_visualization.py --checkpoint checkpoints/best.pt --device cuda
display(Image("outputs/figures/fig7_attention_visualization.png"))

## 7. Ablation study — one group per cell (each run resumes automatically; finished runs are skipped)
If Colab times out in the middle of a cell: run CELL 1 and CELL 2 again, then re-run the same ablation cell.

In [ ]:
s = 200 if QUICK else ABLATION_STEPS
extra = "--set model.d_model=128 model.ffn_dim=512" if ABLATION_SMALL else ""
!python experiments/ablation.py --group depth --steps {s} {extra}

In [ ]:
!python experiments/ablation.py --group heads --steps {s} {extra}

In [ ]:
!python experiments/ablation.py --group context --steps {s} {extra}

In [ ]:
# optional extras (run only if time permits)
# !python experiments/ablation.py --group position --steps {s} {extra}
!python experiments/ablation.py --group generation --checkpoint checkpoints/best.pt
for g in ["depth", "heads", "context"]:
    display(Image(f"outputs/figures/fig9_ablation_{g}.png"))
print(open("outputs/ablation/ablation_results.md").read())

## 8. API smoke test (same app that is deployed on Render)

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "src.api:app", "--port", "8000"])
time.sleep(8)
print(requests.get("http://127.0.0.1:8000/health").json())
r = requests.post("http://127.0.0.1:8000/generate", json={"prompt": "ROMEO:", "max_new_tokens": 120, "temperature": 0.8})
print(r.json()["generated"])
server.terminate()

## 9. Download results (they are also already in Drive → MyDrive/minigpt_runs)

In [ ]:
!zip -qr results.zip outputs/. checkpoints/best.pt
from google.colab import files
files.download("results.zip")